In [34]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR, GREEDY_CONFIG
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form
from src.models import get_messages
from tqdm import tqdm


### Choose the prompting configuration

In [35]:
filename = "2002SCC33"
split = "dev"
filepath = Path(DATA_DIR) / "original" / split / f"{filename}.html"
filepath = Path("output")  / f"{filename}_0.html"

filepath = PROJECT_ROOT / Path("output") / "dev_gpt5.2_DEC_fs6_greedy_sentence_long" / filename / f"{filename}_2.html"
with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()



In [36]:
### Choose the right worflow
method = "DEC3" # "AIO" | "DEC0" | "DEC1" | "DEC2" | "DEC3"

if method == "AIO":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources", "title", "citation", "source", "authors", "fragment"]

    spans_in_context = True

    prompt_filename = "allInOne_long.txt"


if method == "DEC0":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources"]

    spans_in_context = True

    prompt_filename = "decomposed0_long.txt"

if method == "DEC1":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources"]
    new_labels = ["title", "fragment"]

    spans_in_context = False


    prompt_filename = "decomposed1-3.txt"

if method == "DEC2":
    parents = ["secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment"]
    new_labels = ["source", "authors"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

if method == "DEC3":
    parents = ["decision", "legislation"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment", "source", "authors"]
    new_labels = ["citation"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

#### Common Few SHot Selection

In [37]:
fewshot_method = "greedy"   # "greedy" | "random"

fewshot_filename = f"examples_{fewshot_method}_surf-{GREEDY_CONFIG['surface_pattern']}_struct-{GREEDY_CONFIG['structural_pattern']}"
if fewshot_method == "greedy":
    with open(FEWSHOT_CACHE_DIR / f"{fewshot_filename}.json", "r", encoding="utf-8") as f:
        fewshot_file_content = json.load(f)

fewshot_examples = [(example["example"]["input"], example["example"]["output"]) for example in fewshot_file_content["examples"]]
print("fewshot examples from :", fewshot_filename)


fewshot examples from : examples_greedy_surf-1.0_struct-0.0


##### Few Shot processing step

In [38]:
nb_fewshot_examples = 6
allowed_labels = already_labeled_labels + new_labels

input_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
)

output_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels + new_labels,
    keep_attributes=["labelname"]
)


# Transform the output in it simplified form
final_fewshot = []
total_output_text = ""
for example in fewshot_examples:
    input, output = example

    input_tokens = tokenize(input)
    transformed_input_tokens = prepare_label_tokens(input_tokens, input_label_config)

    output_tokens = tokenize(output)
    transformed_output_tokens = prepare_label_tokens(output_tokens, output_label_config)

    if spans_in_context:
        final_fewshot.append((decode(transformed_input_tokens), decode(transformed_output_tokens)))

    if not spans_in_context:

        total_output_text += "|||" + decode(transformed_output_tokens)

final_fewshot = final_fewshot[:nb_fewshot_examples]


if not spans_in_context:
    parents_dict = _parse_parent_annotations(total_output_text)
    for parent_name, annotations in parents_dict.items():
        if parent_name not in parents:
            continue
        for annotation in annotations:
            input = decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))

            if input != annotation:
                final_fewshot.append((input, annotation))

#### Common Prompt loading

In [39]:
from src.prompts.prompt_utils import build_sublabel_definitions

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

from src.prompts.sublabel_definitions import SUBLABEL_DEFINITIONS_V2


if method in ["DEC1", "DEC2", "DEC3"]:
    sublabels_str = ", ".join(new_labels)
    sublabels_definition = build_sublabel_definitions(set(new_labels) - set(parents), sublabel_definitions=SUBLABEL_DEFINITIONS_V2)

    system_prompt = system_prompt.format(
            sublabels=sublabels_str,
            sublabels_definition=sublabels_definition,
        )

system_prompt used :  decomposed1-3.txt


#### Assistant loading

In [14]:
from src.main import MODEL_MAPPING_NAME
from src.models import AssistantFactory

model = "gpt-5.2"

if model == "gpt-5.2":
        assistant = AssistantFactory.create_from_config({
            "type": "openai",
            "model_name": model,
            "temperature": 1,
        })
else:
    assistant = AssistantFactory.create(MODEL_MAPPING_NAME[model])

    
#gpt5_2_config= {
#        "type": "openai",
#        "model_name": "gpt-5.2",
#        "temperature": 1,
#    }

#assistant = AssistantFactory.create_from_config(gpt5_2_config)


# SaulLM-7B-Instruct
# Qwen2.5-7B-Instruct
# Qwen2.5-32B-Instruct
#assistant = AssistantFactory.create("Qwen2.5-7B-Instruct")

#### Chunk output controle

In [28]:
def process_output(generated, token_chunk, allowed_labels, assistant, with_fallback: bool = True):

    from src.output_control.processor import OutputProcessor
    from src.output_control.fallback import FallbackHandler 
    from src.output_control.verification import VerificationResult 

    controller = OutputProcessor()
    fallback_handler = FallbackHandler(processor=controller)
    
    corrected_generated_tokens, status = controller.process(
        raw_llm_output=generated,
        original_chunk=token_chunk,
        allowed_labels=allowed_labels
    )

    if status.passed:
        return corrected_generated_tokens, status

    if not with_fallback:
        return token_chunk, status
    
    
    corrected_generated_tokens, status_dict = fallback_handler.handle_failure(
        assistant=assistant,
        corrected_output=corrected_generated_tokens,
        original_chunk=token_chunk,
        initial_status=status,
        allowed_labels=allowed_labels,
        fallback_prompt_filename="fallback.txt"
    )
    # Convert dict to VerificationResult
    status = VerificationResult(
        passed=status_dict.get('passed', False),
        error_type=status_dict.get('error_type'),
        details=status_dict.get('error_details'),
        tokens=corrected_generated_tokens
    )
    
    return corrected_generated_tokens, status

### For AIO or DEC0 ONLY

#### Chunking with the chunker

In [46]:
chunker = "sentence"  # "paragraph" | "sentence"

from src.chunkers.cache import cache_exists, load_cache
from src.chunkers import ChunkerFactory

if not cache_exists(chunker, split, filename):

    # Load spaCy only if needed
    nlp = None
    if chunker == "sentence":
        import spacy
        nlp = spacy.load("en_core_web_trf")
        print("✅ Model loaded.\n")


    token_chunks = ChunkerFactory.get_chunks(
        html_content, method=chunker, split=split, filename=filename, nlp=nlp
    )

else:
    token_chunks = load_cache(chunker, split, filename)




#### Test for SaulLM

In [10]:
token_chunk = token_chunks[0]


In [11]:
user_input =  decode(token_chunk)
print("User input : ", user_input)

User input :   Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the Excise Tax Act,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the Excise Tax Act. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the Excise Tax Act. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J. Martin
Secretary UNOFFICIAL
SUMMARY Appeal
No. 2845 CAN
TRAFFIC S

In [12]:
messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot[:1], has_system_role=True)


In [50]:
def _format_saul_prompt(messages: list) -> str:
    """
    Formats messages for SaulLM-7B (Llama-2 based).
    System message is injected into the <<SYS>> block of the first user turn.
    """
    system_content = ""
    conversation = []
    
    for msg in messages:
        if msg["role"] == "system":
            system_content = msg["content"]
        else:
            conversation.append(msg)
    
    prompt = "<s>"
    
    for i, msg in enumerate(conversation):
        if msg["role"] == "user":
            prompt += "[INST] "
            # Inject system prompt into the first user turn only
            if i == 0 and system_content:
                prompt += f"<<SYS>>\n{system_content}\n<</SYS>>\n\n"
            prompt += f"{msg['content']} [/INST]"
        elif msg["role"] == "assistant":
            prompt += f" {msg['content']} </s><s>"
    
    return prompt

In [51]:
print(messages)


[{'role': 'system', 'content': 'Context\nIn Canadian decisions, legal sources (legislation, decisions, or secondary sources) are cited to support reasoning. References may be full citations, short forms, abbreviations, or contextual clues (e.g., "the Act").\n\nRole\nAnnotate legal source references in Canadian legal decisions.\n\nLegal Source Types\n<legislation> – Statutes, regulations, constitutions, treaties (e.g., Criminal Code, s. 8 of the Charter)\n<decision> – Court or tribunal decisions (e.g., R. v. Jordan, 2016 SCC 27)\n<secondary sources> – Scholarship, commentaries, journal articles, legal dictionaries (e.g., Driedger, Sullivan)\n\nScope\nIdentify all mentions, including: first and subsequent mentions, full/short/abbreviated citations, contextual references ("the Act"), Latin terms (ibid., supra), and footnotes.\n\nLabel Rules\n- One label per mention\n- Label exact text span\n- Do not merge distinct sources\n- Do not invent sources\n- Prioritize precision over recall\n\nYou

In [ ]:
print(messages[0]["content"])

Annotate this text: Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the Excise Tax Act,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the Excise Tax Act. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the Excise Tax Act. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J. Martin
Secretary UNOFFICIAL
SUMMARY Appeal
No. 2845 CAN
TRAF

In [ ]:
import torch
def generate(
        assistant,
        messages,  # Full conversation history
        max_new_tokens: int = 512,
    ) -> str:
        """
        Generate a response for Qwen2.5-7B-Instruct.
        
        Args:
            messages: List of dicts with "role" and "content" keys.
                    Example: [
                        {"role": "system", "content": "You are..."},
                        {"role": "user", "content": "Hello"},
                        {"role": "assistant", "content": "Hi!"},
                        {"role": "user", "content": "Another question"}
                    ]
            max_new_tokens: Maximum tokens to generate.
        """
        # Apply chat template - Qwen2.5 handles everything automatically
        text = assistant._format_saul_prompt(messages)
        
        inputs = assistant.tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False  # <-- important: we added <s> manually
        ).to(assistant.model.device)
        
        # Recommended sampling parameters for Qwen2.5-Instruct[citation:4][citation:9]
        with torch.no_grad():
            outputs = assistant.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=assistant.temperature,
                do_sample=True,
                repetition_penalty=1.05,
                pad_token_id=assistant.tokenizer.eos_token_id,
            )
        
        # Decode only the newly generated tokens
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = assistant.tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        return response

In [13]:
generated = assistant.generate(messages=messages)


/home/zagar/myenv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [143]:
generated = generate(assistant=assistant, messages=message)


In [14]:
print(generated)

Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the <legislation>Excise Tax Act</legislation>,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the <legislation>Excise Tax Act</legislation>. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the <legislation>Excise Tax Act</legislation>. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J.

In [15]:
corrected_generated_tokens, status = process_output(generated, token_chunk=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


In [16]:
print(decode(corrected_generated_tokens))

 Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the <auto_label labelname="legislation">Excise Tax Act</auto_label>,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the <auto_label labelname="legislation">Excise Tax Act</auto_label>. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the <auto_label labelname="legislation">Excise Tax Act</auto_label>. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.

In [ ]:
user_input =  decode(token_chunk)

message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=True)

generated = assistant.generate(message=message)

corrected_generated_tokens, status = process_output(generated, token=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


processed_chunks.append(corrected_generated_tokens)

#### Main processing function

In [47]:
from src.models import get_messages
from tqdm import tqdm

processed_chunks = []
for token_chunk in tqdm(token_chunks):

    user_input =  decode(token_chunk)

    message = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=assistant.has_system_role)

    generated = assistant.generate(messages=message)
    #print("Generated : ", generated)
    corrected_generated_tokens, status = process_output(generated, token_chunk=token_chunk, allowed_labels=allowed_labels, assistant=assistant)
    #print("Corrected generated tokens : ", decode(corrected_generated_tokens))

    processed_chunks.append(corrected_generated_tokens)

100%|██████████| 17/17 [01:44<00:00,  6.17s/it]


#### Post Processing

In [48]:
from src.post_processing.main import tokens_to_html
from src.post_processing.token_operations import flatten_token_chunks

processed_tokens_flat = flatten_token_chunks(processed_chunks)

output_html_content = tokens_to_html(processed_tokens_flat, html_content)

   ✓ Flattened 17 chunks into 10786 tokens
   ✓ Merged to 14812 tokens
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Corrected 15034 tokens
   ✓ Brackets are coherent

✓ POST-PROCESSING COMPLETE
Final HTML length: 128989 characters



#### File saving

In [49]:
output_filename = PROJECT_ROOT / "output" / f"{filename}_0.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(output_html_content)

### For DEC1-3 

No chunking needed here, we just need the list of mention already labeled

#### Convert into tokens

In [40]:
from src import extract_body, tokenize, clean_tokens
tokens = tokenize(html_content)


#### Get already extracted mention

In [41]:

from src.extractor import build_processing_segments
from src.extractor import get_list_of_mention
from src.models import get_messages
from tqdm import tqdm
from tqdm import tqdm

parent_mentions = get_list_of_mention(
        tokens=tokens,
        keep_labels=parents,
        label_type="auto_label"  # Process auto_labels from parent extraction
    )

print(f"Found {len(parent_mentions)} parent mentions to process")

segments = build_processing_segments(tokens, parent_mentions)

print(f"Built {len(segments)} token segments "
        f"({sum(s['process'] for s in segments)} to process)")

Found 278 parent mentions to process
Built 557 token segments (278 to process)


#### nesting tag test TBR

In [77]:
html = "blabla blabla <auto_label labelname='legislation'><span lang='EN-CA'><i>Crane Canada Inc. v. Sécurité nationale, compagnie d’assurances and Attorney General of </i></span><i><span lang='EN-CA'>Quebec</span></i></decision></auto_label> ok okokok ok"
soup = BeautifulSoup(html, 'html.parser')
soup = fix_labels(html)
print(soup)

blabla blabla <span lang="EN-CA"><i><auto_label labelname="legislation">Crane Canada Inc. v. Sécurité nationale, compagnie d’assurances and Attorney General of Quebec</auto_label></i></span> ok okokok ok


In [78]:
html = "blabla blabla <auto_label labelname='secondary sources'><i><span lang='EN-GB' style='font-size:12.0pt; letter-spacing:-.1pt'>Webster's Third New International Dictionary of the English Language Unabridged</span></i><span lang='EN-GB' style='font-size:12.0pt;letter-spacing:-.1pt'> (1979)</span></auto_label> ok okokok ok"
soup = BeautifulSoup(html, 'html.parser')
soup = fix_labels(html)
print(soup)

blabla blabla <span lang="EN-GB" style="font-size:12.0pt; letter-spacing:-.1pt"><auto_label labelname="secondary sources"><i>Webster's Third New International Dictionary of the English Language Unabridged</i> (1979)</auto_label></span> ok okokok ok


In [79]:
html = "blabla blabla <auto_label labelname='secondary sources'><span class='MsoFootnoteReference'><span class='MsoFootnoteReference'>[6]</span></span></auto_label> ok okokok ok"
soup = BeautifulSoup(html, 'html.parser')
soup = fix_labels(html)
print(soup)

blabla blabla <span class="MsoFootnoteReference"><span class="MsoFootnoteReference"><auto_label labelname="secondary sources">[6]</auto_label></span></span> ok okokok ok


In [86]:
for segment in segments:
    if not segment["process"]:
        continue
    
    mention = segment["tokens"]
    html_label = segment["meta"]["label"]

    
    prepared_tokens = prepare_label_tokens(mention, input_label_config)
    user_input = decode(prepared_tokens)

    print(user_input)

<legislation>section 51.19 of the <i>Excise Tax Act</i>,
R.S.C. 1970, c. E-13</legislation>
<legislation>section 51.17 of the <i>Excise Tax Act</i></legislation>
<legislation>paragraph 1<i>(h)</i>, Part XII,
Schedule III of the <i>Excise Tax Act</i></legislation>
<legislation>paragraph 1(h), Part XII, Schedule III of the Excise Tax Act</legislation>
<secondary sources>Driedger,
E.A., <u>Construction of Statutes</u> (Second Edition)</secondary sources>
<legislation><i>Excise Tax
Act</i><a href="#_ftn1" name="_ftnref1" title=""><span class="MsoFootnoteReference"><span class="MsoFootnoteReference"><span lang="EN-GB" style='font-size:12.0pt;font-family:"Times New Roman";letter-spacing:-.1pt'>[1]</span></span></span></a>
(the Act)</legislation>
<legislation>LEGISLATION</legislation>
<legislation>27(1) <i>There shall be imposed, levied and
collected a consumption or sales  tax ... on the sale price of all goods</i></legislation>
<legislation>29(1) The tax imposed by section 27 does
not apply

In [93]:
from src.prompts.prompt_utils import build_sublabel_definitions

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

from src.prompts.sublabel_definitions import SUBLABEL_DEFINITIONS_V2


if method in ["DEC1", "DEC2", "DEC3"]:
    sublabels_str = ", ".join(new_labels)
    sublabels_definition = build_sublabel_definitions(set(new_labels) - set(parents), sublabel_definitions=SUBLABEL_DEFINITIONS_V2)

    system_prompt = system_prompt.format(
            sublabels=sublabels_str,
            sublabels_definition=sublabels_definition,
        )

system_prompt used :  decomposed1-3.txt


#### Main processing function

In [42]:
config = LabelTransformConfig(
    use_simplified=False,
    switch_type=False,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
) # We only remove the attribute
 

failed_count = 0

for idx, segment in enumerate(tqdm(segments, desc="Processing mentions")):
        if not segment["process"]:
            continue


        mention = segment["tokens"]
        html_label = segment["meta"]["label"]

        
        prepared_tokens = prepare_label_tokens(mention, input_label_config)
        user_input = decode(prepared_tokens)

        filtered_fewshot = []
        for example in final_fewshot:
            if example[0].startswith(f"<{html_label.name}>"):
                filtered_fewshot.append(example)
        messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=filtered_fewshot, has_system_role=assistant.has_system_role)


        generated = assistant.generate(messages=messages)
        #print(generated)
        
        corrected_generated_tokens, status = process_output(generated=generated, token_chunk=prepare_label_tokens(mention, config), allowed_labels=allowed_labels, assistant=assistant, with_fallback=False)
        #print(decode(corrected_generated_tokens))
        if not status.passed:
            failed_count += 1

        segment["tokens"] = corrected_generated_tokens

processed_tokens = [
        token
        for segment in segments
        for token in segment["tokens"]
    ]

Processing mentions:   0%|          | 0/557 [00:00<?, ?it/s]

Processing mentions: 100%|██████████| 557/557 [07:46<00:00,  1.19it/s]


#### Post Processing : tokens to HTML

In [43]:
from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
processed_html_content = tokens_to_html_after_decomposed1_3_prompting(processed_tokens, html_content)

   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)


#### Save File

In [45]:
output_filename = PROJECT_ROOT / Path("output") / "dev_gpt5.2_DEC_fs6_greedy_sentence_long" / filename / f"{filename}_final.html"
#output_filename = Path("output") / f"{filename}_1.html"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(processed_html_content)